# 04 — Submission Sanity Check

Verify submission zip before uploading to Codabench.  
Checks: file structure, field names, image_id range, score distribution, position bounds.

In [ ]:
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Point to your submission zip
SUBMISSION_ZIP = '../work_dirs/submission_m960_test/submission.zip'

# Which split? test=9309, challenge=11352
EXPECTED_SPLIT = 'test'  # or 'challenge'
EXPECTED_IMAGES = {'test': 9309, 'challenge': 11352}[EXPECTED_SPLIT]
MAX_IMAGE_ID = EXPECTED_IMAGES - 1

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP) as zf:
    names = zf.namelist()
    print(f'Files in zip: {names}')
    
    assert 'results.json' in names, 'MISSING: results.json'
    assert 'metadata.json' in names, 'MISSING: metadata.json'
    print('Structure OK')
    
    with zf.open('metadata.json') as f:
        meta = json.load(f)
        print(f'\nMetadata: {meta}')
        assert 'score_threshold' in meta, 'MISSING: score_threshold in metadata'
    
    with zf.open('results.json') as f:
        results = json.load(f)

print(f'\nTotal detections: {len(results)}')

In [ ]:
# Check required fields
r0 = results[0]
required_keys = {'image_id', 'category_id', 'position_on_pitch', 'score'}
actual_keys = set(r0.keys())
missing = required_keys - actual_keys
extra = actual_keys - required_keys - {'area'}

print(f'Keys: {actual_keys}')
if missing:
    print(f'MISSING keys: {missing}')
if extra:
    print(f'EXTRA keys (might cause issues): {extra}')
if not missing:
    print('All required keys present')

print(f'\nSample detection:')
print(json.dumps(r0, indent=2))

In [ ]:
# Check image_id range
img_ids = sorted(set(r['image_id'] for r in results))
print(f'Unique image_ids: {len(img_ids)}')
print(f'Range: {min(img_ids)} - {max(img_ids)} (expected 0 - {MAX_IMAGE_ID})')

invalid = [i for i in img_ids if i > MAX_IMAGE_ID]
if invalid:
    print(f'INVALID image_ids (>{MAX_IMAGE_ID}): {invalid[:10]}...')
    print('THIS WILL CAUSE AssertionError ON CODABENCH!')
else:
    print(f'All image_ids within valid range for {EXPECTED_SPLIT} split')

In [ ]:
# Score distribution
scores = [r['score'] for r in results]
print(f'Score: min={min(scores):.4f}, median={np.median(scores):.4f}, max={max(scores):.4f}')

# Position bounds
xs = [r['position_on_pitch'][0] for r in results]
ys = [r['position_on_pitch'][1] for r in results]
print(f'X range: {min(xs):.1f} to {max(xs):.1f} (pitch: -52.5 to 52.5)')
print(f'Y range: {min(ys):.1f} to {max(ys):.1f} (pitch: -34 to 34)')

# Detections per image
from collections import Counter
dets_per_img = Counter(r['image_id'] for r in results)
counts = list(dets_per_img.values())
print(f'\nDets/image: min={min(counts)}, median={np.median(counts):.0f}, max={max(counts)}')
print(f'Images with 0 dets: {EXPECTED_IMAGES - len(img_ids)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(scores, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Score')
axes[0].set_title('Score distribution')

axes[1].hist(counts, bins=range(0, max(counts)+2), edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Detections per image')
axes[1].set_title('Detections per image')

# Pitch heatmap
axes[2].hist2d(xs, ys, bins=50, cmap='YlOrRd')
axes[2].set_xlabel('X (m)')
axes[2].set_ylabel('Y (m)')
axes[2].set_title('Position heatmap')
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

print('\nSanity check complete. If no errors above, safe to upload to Codabench.')